In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

random_state = 67
np.random.seed(random_state)

In [ ]:
url = ''
df = pd.DataFrame(url)

df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape[0]-df.dropna().shape[0]

In [ ]:
df.isna().sum()

In [ ]:
df.boxplot(figsize=(15,12))
plt.show()

In [ ]:
plt.figure(figure=(15,12))
sns.heatmap(df.corr(),annot=True)

In [ ]:
df.nunique()

# preprocessing

In [ ]:
df = df.dropna()

In [ ]:
from sklearn.preprocessing import LabelEncoder
column_to_encode = ''
le = LabelEncoder()
df[column_to_encode] = le.fit_transform(column_to_encode)


In [ ]:
from sklearn.preprocessing import OneHotEncoder
one = OneHotEncoder()
column_to_encode = ''
enc_data = one.fit_transform(df[column_to_encode])
categories = list(one.categories_[0])
enc_df = pd.DataFrame(enc_data.toarray(),columns=categories)
df = df.join(enc_df)
df = df.drop(column_to_encode, axis=1)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
categories = []
column_to_encode = ''
oe = OrdinalEncoder(categories=categories,dtype=int)
df[column_to_encode] = oe.fit_transform(df[column_to_encode])

In [ ]:
from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler()
df_scaled = pd.DataFrame(mms.fit_transform(df),columns=df.columns)

In [ ]:
from sklearn.preprocessing import PowerTransformer,StandardScaler
from sklearn.pipeline import make_pipeline
pipeline = make_pipeline(PowerTransformer(),StandardScaler())
df_standardize = pd.DataFrame(pipeline.fit_transform(df_scaled),columns=df.columns)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
df_transformed = pca.fit_transform(df)
min_variance = 0.9
variance_cumsum = np.cumsum(pca.explained_variance_ratio_)
cuttoff_index = np.argmax(variance_cumsum>=min_variance)
X_transformed = df_transformed[:,:cuttoff_index+1]

# training

In [ ]:
X = df
n_clusters = [*range(2,10)]

## Kmean

In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

results_km = pd.DataFrame([],columns=('model','n_clusters','inertia','silhoette_score'))

param_km = {'n_clusters':n_clusters}
gs_km = ParameterGrid(param_grid=param_km)

for param in gs_km:
    km = KMeans(n_clusters=param['n_clusters'],random_state=random_state)
    y_km = km.fit_transform(df)

    results_km.loc[len(results_km)] = [
        f'KMean with {param['n_clusters']} cluster',
        param['n_clusters'],
        km.inertia_,
        silhouette_score(X,y_km)
    ]


In [ ]:
fig,ax = plt.subplots()

ax.plot(n_clusters,results_km['inertia'],color='red')
ax.set_xlabel('n_clusters')
ax.set_ylabel('inertia',color='red')

ax2 = ax.twinx()
ax2.plot(n_clusters,results_km['silhouette'],color='blue')
ax2.set_ylabel('silhouette_score')
ax2.set_ylim(0,1)

plt.show()

## Agglomerative

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import silhouette_score

results_agl = pd.DataFrame([],columns=['n_clusters','silhouette_score'])

param_agl = {'n_clusters':[*range(2,7)]}
pg_agl = ParameterGrid(param_agl)

for param in pg_agl:
    agl = AgglomerativeClustering(n_clusters=param['n_clusters'])
    y_agl = agl.fit_predict(X)
    results_agl.loc[len(results_agl)] = [
        param['n_clusters'],
        silhouette_score(X,y_agl)
    ]


display(results_agl)

## DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

results_dbs = pd.DataFrame([],columns=['eps','min_samples','n_clusters','silhouette_score','unclust%'])

param_dbs = {'eps':[*range(0.001,1,0.05)],'min_samples':[*range(2,10)]}
pg_dbs = ParameterGrid(param_dbs)

for param in pg_dbs:
    dbs = DBSCAN(eps=param['eps'],min_samples=param['min_samples'])
    y_dbs = dbs.fit_predict(X)
    clusterd_elements = X[y_dbs != -1]
    clusterd_labels = y_dbs[y_dbs != -1]
    n_clusters = np.unique(clusterd_labels).shape[0]
    unclust = 1 - clusterd_elements.shape[0] / X.shape[0]

    results_dbs.loc[len(results_dbs)] = [
        param['eps'],
        param['min_samples'],
        n_clusters,
        silhouette_score(clusterd_elements,clusterd_labels),
        unclust*100
    ]



# Display results

In [ ]:
cluster_size_km = np.unique(y_km,return_counts=True)
pd.DataFrame(cluster_size_km[1]).plot.pie(y=0,autopct='%1.1f%%')

In [ ]:
X['clusters'] = y_km
sns.pairplot(X,hue='clusters')